# 09 — Deterministic Agent Policy

This notebook demonstrates the current deterministic research policy using synthetic inputs only. It connects `tools/agent_policy.py` (POLICY_VERSION 1.1) to `src/runtime/deterministic_safety.py` and the canonical frozen threshold metadata.

It does not load model weights, read MURA images or labels, run fine-tuning, call an LLM, or execute TTA/Grad-CAM. Optional explanation tools are selected by policy but never executed here. Every example remains research-only and requires human review.

## 1. Locate the repository and read the frozen policy threshold

In [ ]:
import importlib.util
import json
import os
from pathlib import Path
import sys

def locate_project_root():
    configured = os.environ.get("MURA_PROJECT_ROOT")
    candidates = [Path(configured).resolve()] if configured else []
    here = Path.cwd().resolve()
    candidates.extend([here, *here.parents])
    for candidate in candidates:
        if (candidate / "CHARTER.md").is_file() and (candidate / "tools" / "agent_policy.py").is_file():
            return candidate
    raise FileNotFoundError("Set MURA_PROJECT_ROOT to the repository root.")

PROJECT_ROOT = locate_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

spec = importlib.util.spec_from_file_location("agent_policy", PROJECT_ROOT / "tools" / "agent_policy.py")
agent_policy = importlib.util.module_from_spec(spec)
spec.loader.exec_module(agent_policy)
from src.runtime import deterministic_safety as safety

bundle = json.loads((PROJECT_ROOT / "configs" / "model_b_artifact_bundle.json").read_text(encoding="utf-8"))
assert bundle["bundle_id"] == "checkpoint1-model-b-mean-probability-95-sensitivity"
assert agent_policy.POLICY_VERSION == "1.1"
THRESHOLD = float(bundle["model"]["threshold"])
print({"policy_version": agent_policy.POLICY_VERSION, "threshold_source": "canonical Model B bundle metadata", "threshold": THRESHOLD, "model_loaded": False})

## 2. Build a synthetic, internally consistent Agent record

In [ ]:
def make_record(probabilities, threshold, study_id="synthetic-study"):
    evidence = safety.view_evidence(probabilities, threshold)
    predicted = int(evidence["mean_probability"] >= threshold)
    action = "priority_human_review" if predicted == 1 or evidence["threshold_disagreement"] else "routine_workflow"
    return {
        "study_id": study_id,
        "workflow_status": "SUCCESS",
        "qc_status": "PASS",
        "anatomy_status": "SUPPORTED_WRIST",
        "classification_status": "SUCCESS",
        "classification_performed": True,
        "classification_valid": True,
        "classification_modified": False,
        "predicted_class": predicted,
        "study_probability": evidence["mean_probability"],
        "classification_threshold": threshold,
        "agent_action": action,
        "image_predictions": list(probabilities),
        "view_consistency": {
            "status": "SUCCESS",
            "image_count": evidence["view_count"],
            "min_probability": evidence["min_probability"],
            "max_probability": evidence["max_probability"],
            "mean_probability": evidence["mean_probability"],
            "probability_range": evidence["probability_range"],
            "probability_std": evidence["probability_std"],
            "threshold_disagreement": evidence["threshold_disagreement"],
            "image_classes": [int(p >= threshold) for p in probabilities],
            "threshold": threshold,
        },
    }

priority_record = make_record([THRESHOLD - 0.10, THRESHOLD + 0.10], THRESHOLD)
routine_record = make_record([THRESHOLD - 0.15, THRESHOLD - 0.10], THRESHOLD, "synthetic-routine-study")
priority_decision = agent_policy.select_agent_tools(priority_record, THRESHOLD)
routine_decision = agent_policy.select_agent_tools(routine_record, THRESHOLD)
assert priority_decision["policy_status"] == "SUCCESS"
assert priority_decision["selected_tools"] == agent_policy.OPTIONAL_TOOLS
assert priority_decision["requires_human_review"] is True
assert routine_decision["policy_status"] == "SUCCESS"
assert routine_decision["selected_tools"] == []
assert routine_decision["requires_human_review"] is True
print({"priority": priority_decision, "routine": routine_decision})

## 3. Invalid policy records fail closed

In [ ]:
invalid_record = dict(priority_record)
invalid_record["classification_modified"] = True
blocked = agent_policy.select_agent_tools(invalid_record, THRESHOLD)
assert blocked["policy_status"] == "SAFETY_BLOCK"
assert blocked["selected_tools"] == []
assert blocked["requires_human_review"] is True
print({"invalid_record": blocked, "result": "PASS"})

## 4. Deterministic safety layer: synthetic QC and anatomy

In [ ]:
import io
import numpy as np
from PIL import Image

buffer = io.BytesIO()
gradient = np.tile(np.arange(256, dtype=np.uint8), (256, 1))
Image.fromarray(gradient).save(buffer, format="PNG")
synthetic_bytes = buffer.getvalue()
qc = safety.technical_qc(synthetic_bytes, image_id="synthetic-view")
wrist = safety.validate_declared_anatomy("XR_WRIST")
unsupported = safety.validate_declared_anatomy("XR_HAND")
assert qc["status"] == "PASS"
assert wrist["status"] == "SUPPORTED_WRIST"
assert unsupported["status"] == "UNSUPPORTED"
print({"synthetic_qc": qc["status"], "wrist": wrist["status"], "hand": unsupported["status"], "raw_data_read": False})

## Interpretation boundary

`select_agent_tools` validates a proposed record and selects optional research explanation tools; it does not run those tools or change model predictions. The safe route cannot clear a study from human review. The threshold comes from the canonical bundle metadata, while the weights remain at `DATA/MODEL_B_FINE_TUNING/` and are not loaded in this synthetic demonstration.

This is a policy demonstration, not an agent-performance evaluation or clinical validation.